# Exploratory Data Analysis
**Walmart Sales Forecasting & Inventory Optimization**

Covers:
- Sales trends over time
- Seasonality patterns (monthly heatmap)
- Distribution analysis
- Store and department comparisons
- Holiday effect analysis
- Time-series decomposition
- Stationarity tests (ADF, KPSS)

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
import warnings
warnings.filterwarnings('ignore')

from data.data_loader import load_config, load_processed_data

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)

config = load_config('../config/config.yaml')
df = load_processed_data(config)
print(df.shape)
df.head()

## 1. Overall Sales Trend

In [ ]:
weekly_total = df.groupby('Date')['Weekly_Sales'].sum().reset_index()

fig, ax = plt.subplots()
ax.plot(weekly_total['Date'], weekly_total['Weekly_Sales'], linewidth=1.5, color='steelblue')
ax.fill_between(weekly_total['Date'], weekly_total['Weekly_Sales'], alpha=0.15, color='steelblue')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax.set_title('Total Weekly Sales Across All Stores & Departments', fontsize=14)
ax.set_xlabel('Date'); ax.set_ylabel('Total Weekly Sales ($)')
plt.tight_layout(); plt.show()

## 2. Monthly Seasonality Heatmap

In [ ]:
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

monthly_pivot = df.pivot_table(values='Weekly_Sales', index='Year', columns='Month', aggfunc='sum')
monthly_pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(monthly_pivot, annot=True, fmt='.2e', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Monthly Sales Heatmap (Total $)', fontsize=14)
plt.tight_layout(); plt.show()

## 3. Sales Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df['Weekly_Sales'], bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_title('Distribution of Weekly Sales')
axes[0].set_xlabel('Weekly Sales ($)')

df.boxplot(column='Weekly_Sales', by='Month', ax=axes[1], patch_artist=True)
axes[1].set_title('Weekly Sales by Month (Boxplot)')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Weekly Sales ($)')
plt.suptitle('')
plt.tight_layout(); plt.show()

## 4. Top 10 Stores by Total Sales

In [ ]:
store_totals = df.groupby('Store')['Weekly_Sales'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 4))
store_totals.plot(kind='bar', ax=ax, color='teal', edgecolor='white')
ax.set_title('Top 10 Stores by Total Sales')
ax.set_xlabel('Store'); ax.set_ylabel('Total Sales ($)')
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

## 5. Holiday Effect

In [ ]:
holiday_avg = df.groupby('IsHoliday')['Weekly_Sales'].mean().reset_index()
holiday_avg['IsHoliday'] = holiday_avg['IsHoliday'].map({True: 'Holiday Week', False: 'Regular Week'})

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=holiday_avg, x='IsHoliday', y='Weekly_Sales', palette=['#4c72b0','#dd8452'], ax=ax)
ax.set_title('Average Weekly Sales: Holiday vs Regular Weeks')
ax.set_xlabel('')
ax.set_ylabel('Avg Weekly Sales ($)')
plt.tight_layout(); plt.show()

## 6. Time-Series Decomposition (Store 1, Dept 1)

In [ ]:
series = df[(df['Store'] == 1) & (df['Dept'] == 1)].set_index('Date')['Weekly_Sales'].sort_index()
series = series.fillna(method='ffill')

decomp = seasonal_decompose(series, model='additive', period=52)
fig = decomp.plot()
fig.set_size_inches(14, 8)
plt.suptitle('Seasonal Decomposition — Store 1, Dept 1', y=1.01, fontsize=14)
plt.tight_layout(); plt.show()

## 7. Stationarity Tests

In [ ]:
def adf_test(series, name='Series'):
    result = adfuller(series.dropna())
    print(f'\n--- ADF Test: {name} ---')
    print(f'ADF Statistic : {result[0]:.4f}')
    print(f'p-value       : {result[1]:.4f}')
    print(f'Critical Values: 1%={result[4]["1%"]:.3f}, 5%={result[4]["5%"]:.3f}')
    if result[1] < 0.05:
        print('Conclusion: Stationary ✓ (reject H0)')
    else:
        print('Conclusion: Non-stationary ✗ (fail to reject H0)')

def kpss_test(series, name='Series'):
    result = kpss(series.dropna(), regression='c', nlags='auto')
    print(f'\n--- KPSS Test: {name} ---')
    print(f'KPSS Statistic: {result[0]:.4f}')
    print(f'p-value       : {result[1]:.4f}')
    if result[1] > 0.05:
        print('Conclusion: Stationary ✓ (fail to reject H0)')
    else:
        print('Conclusion: Non-stationary ✗ (reject H0)')

adf_test(series, 'Store 1 Dept 1')
kpss_test(series, 'Store 1 Dept 1')

# Test first-differenced series
series_diff = series.diff().dropna()
adf_test(series_diff, 'Store 1 Dept 1 (first diff)')